# Predictive Maintenance using Machine Learning

This notebook demonstrates an end-to-end predictive maintenance workflow for industrial equipment using synthetic sensor data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

def generate_synthetic_data(n_samples=5000, random_state=42):
    rng = np.random.default_rng(random_state)
    temperature = rng.normal(82, 7, n_samples)
    vibration = rng.normal(0.55, 0.12, n_samples)
    pressure = rng.normal(75, 6, n_samples)
    humidity = rng.normal(45, 8, n_samples)
    operating_cycles = rng.uniform(50, 1800, n_samples)

    failure_prob = 1 / (
        1
        + np.exp(
            -(
                -6.8
                + 0.095 * (temperature - 70)
                + 6.0 * (vibration - 0.4)
                + 0.06 * (pressure - 70)
                + 0.05 * (humidity - 40)
                + 0.003 * operating_cycles
            )
        )
    )
    failure_prob = np.clip(failure_prob + 0.08 * np.maximum((temperature - 85) / 20, 0), 0.01, 0.95)
    failure = rng.random(n_samples) < failure_prob

    df = pd.DataFrame(
        {
            "temperature": temperature,
            "vibration": vibration,
            "pressure": pressure,
            "humidity": humidity,
            "operating_cycles": operating_cycles,
            "failure": failure.astype(int),
        }
    )
    return df

df = generate_synthetic_data()
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "predictive_maintenance_synthetic.csv"
df.to_csv(csv_path, index=False)
df.head()

## Exploratory Data Analysis

In [ ]:
df.info()
df.describe().T

corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df, x="failure", y="temperature", ax=axes[0, 0])
axes[0, 0].set_title("Temperature vs Failure")
sns.boxplot(data=df, x="failure", y="vibration", ax=axes[0, 1])
axes[0, 1].set_title("Vibration vs Failure")
sns.boxplot(data=df, x="failure", y="pressure", ax=axes[1, 0])
axes[1, 0].set_title("Pressure vs Failure")
sns.boxplot(data=df, x="failure", y="operating_cycles", ax=axes[1, 1])
axes[1, 1].set_title("Operating Cycles vs Failure")
plt.tight_layout()
plt.show()

## Feature Engineering and Model Training

In [ ]:
df["temp_pressure_ratio"] = df["temperature"] / (df["pressure"] + 1e-6)
df["vibration_temp_interaction"] = df["vibration"] * df["temperature"]
df["high_load"] = ((df["temperature"] > 85) & (df["vibration"] > 0.6)).astype(int)

features = [
    "temperature",
    "vibration",
    "pressure",
    "humidity",
    "operating_cycles",
    "temp_pressure_ratio",
    "vibration_temp_interaction",
    "high_load",
]
X = df[features]
y = df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

results = []
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    cm_path = results_dir / f"{name.lower().replace(' ', '_')}_confusion_matrix.png"
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Failure", "Failure"], yticklabels=["No Failure", "Failure"])
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved confusion matrix: {cm_path}")
    if hasattr(model, "feature_importances_"):
        importances = pd.Series(model.feature_importances_, index=features)
        importances = importances.sort_values(ascending=False)
        fi_path = results_dir / f"{name.lower().replace(' ', '_')}_feature_importance.png"
        plt.figure(figsize=(8, 4))
        sns.barplot(x=importances.values, y=importances.index, palette="viridis")
        plt.title(f"{name} Feature Importance")
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.savefig(fi_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved feature importance plot: {fi_path}")
    results.append(
        {
            "model": name,
            "accuracy": round(accuracy_score(y_test, preds), 4),
            "precision": round(precision_score(y_test, preds), 4),
            "recall": round(recall_score(y_test, preds), 4),
            "f1_score": round(f1_score(y_test, preds), 4),
        }
    )
    print(f"\n{name} Classification Report\n")
    print(classification_report(y_test, preds))

metrics_df = pd.DataFrame(results)
metrics_df.sort_values("f1_score", ascending=False)

## Business Interpretation

This model can help maintenance teams identify machines that are likely to fail soon, allowing them to schedule inspections, reduce downtime, and optimize repair costs.